# Traffic Signal OpenEnv: LLM-Based Reinforcement Learning Pipeline

This notebook implements a full RL fine-tuning pipeline using Hugging Face **TRL** and **Unsloth**. It orchestrates grid-level traffic policies by interacting with the OpenEnv HTTP API.

In [ ]:
!pip install -q trl unsloth wandb matplotlib requests numpy

In [ ]:
import os
import requests
import numpy as np
import wandb
import torch
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel

# Initialize WandB
wandb.init(project="traffic-openenv-rl")

ENV_URL = "https://guuru-dev-traffic-signal-openenv-2.hf.space"
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"

In [ ]:
# Load Model with Unsloth optimization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# TRAINING INTEGRITY CHECKLIST (read before running):
# 1. Watch WandB: episode_reward should trend up, mean_queue should trend down
# 2. Inspect printed sample actions every 10 episodes — look for realistic phase decisions
# 3. If the model always outputs the same action regardless of state, stop and debug
# 4. The central controller uses a total_priority_budget constraint — the model cannot
#    boost all intersections simultaneously, so degenerate all-boost policies are blocked
# 5. Reward is multi-component (6 rubrics) — gaming one component cannot dominate score

def reward_fn(prompts, completions, **kwargs):
    """
    Environment-based reward function.
    """
    rewards = []
    for episode, (prompt, completion) in enumerate(zip(prompts, completions), start=1):
        res = requests.post(f"{ENV_URL}/reset", json={"task_id": "hard_multi", "central_enabled": True})
        state = res.json()
        action = completion
        step_res = requests.post(f"{ENV_URL}/step", json={"action": action})
        data = step_res.json()
        info = data.get("info", {})
        reward = float(data.get("reward", 0.0))
        wandb.log({
            "episode_reward": reward,
            "final_score": info.get("final_score", 0.0),
            "throughput": info.get("throughput", 0.0),
            "mean_queue": info.get("mean_queue", 0.0)
        })

        if episode % 10 == 0:
            print(f"\n=== Episode {episode} Sample Inspection ===")
            print(f"  Action sent     : {action}")
            print(f"  Step reward     : {reward:.4f}")
            print(f"  Final score     : {info.get('final_score', 'N/A')}")
            print(f"  Active behaviors: {info.get('active_behaviors_log', [])}")
            print(f"  Mean queue      : {info.get('mean_queue', 'N/A')}")
            print("=" * 48)
            # IMPORTANT: If actions look degenerate (e.g., always KEEP or always SWITCH),
            # stop training and inspect. A rising reward with trivial actions = reward hacking.

        rewards.append(reward)
    return rewards

training_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_prompt_length=512,
    max_completion_length=128,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=None, # Load your traffic scenarios here
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# [Step 5] Save Checkpoint
# Save LoRA adapter weights only (correct Unsloth pattern for 4-bit models)
# Do NOT use model.merge_and_unload() on a 4-bit model — this corrupts weights
model.save_pretrained("outputs/traffic-lora")
tokenizer.save_pretrained("outputs/traffic-lora")
print("Adapter weights saved to outputs/traffic-lora")

# Optional: Push trained adapter to Hugging Face Hub after training
# Uncomment and fill in your token before running
# model.push_to_hub("Guuru-DEV/traffic-signal-agent", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("Guuru-DEV/traffic-signal-agent", token="YOUR_HF_TOKEN")

In [ ]:
# [Step 6] Generate Visualizations
import sys
sys.path.append(os.getcwd())
from env.metrics_exporter import generate_training_plots

# Collect history from trainer/wandb and plot
# generate_training_plots(collected_log, "plots")